# MiniMind-3 Self-Maintaining & Bounded Self-Modifying Loop (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ed2k/minimind/blob/mychanges/selfchange/colab/run_selfchange.ipynb)

Run cells top-to-bottom on a **T4 / L4 / A100 GPU runtime**. State is atomically mirrored to Google Drive every cycle.
To halt cleanly at any time, run `!touch selfchange/workspace/STOP`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
%cd /content
!if [ -d minimind/.git ]; then cd minimind && git fetch origin mychanges && git checkout mychanges && git pull --ff-only origin mychanges; else git clone --depth 1 -b mychanges https://github.com/ed2k/minimind.git; fi
%cd /content/minimind
!pip -q install "transformers==4.57.6" "datasets==3.6.0" "huggingface_hub" "safetensors" "accelerate"
!python3 -m selfchange.kernel.supervisor --check-only

In [ ]:
# Step 1: Acquire MiniMind-3 + anchor splits + record untouched 5-Tier Baseline Scorecard
!python3 -m selfchange.bootstrap.setup_base --mode colab --repo-root . --output-dir selfchange/workspace/models/base
!python3 -m selfchange.bootstrap.eval_baseline --worktree . --backend minimind --model-path selfchange/workspace/models/base/minimind3_768.pth --tokenizer-path model --heldout-path selfchange/workspace/models/base/data/heldout_eval.jsonl --output selfchange/workspace/baseline_untouched.json || true

In [ ]:
# Steps 3-6: Progressive 3-Rung Curriculum SFT + Offline RFT/DPO + Bootstrap-Ready Gate
!python3 -m selfchange.bootstrap.warmstart_sft --work-dir selfchange/workspace/bootstrap_data --anchor-jsonl selfchange/workspace/models/base/data/anchor_sft.jsonl --total-traces 3000 --stage full_sft
!python3 -m selfchange.bootstrap.offline_rft_dpo --backend minimind --model-path selfchange/workspace/models/candidates/warmstart_rung3_768.pth --tokenizer-path model --from-weight warmstart_rung3 --rounds 1 --num-tasks 24 --num-rollouts 4
!python3 -m selfchange.bootstrap.eval_baseline --worktree . --backend minimind --model-path selfchange/workspace/models/candidates/offline_dpo_r1_768.pth --tokenizer-path model --heldout-path selfchange/workspace/models/base/data/heldout_eval.jsonl --output selfchange/workspace/baseline_prepared.json || true

In [ ]:
# Launch re-entrant autonomous loop with Google Drive mirroring
!python3 -m selfchange.colab.colab_launcher --drive-dir /content/drive/MyDrive/minimind_selfchange --model-path selfchange/workspace/models/candidates/offline_dpo_r1_768.pth --tokenizer-path model --max-cycles 10 --autonomy-level 1
!cat selfchange/workspace/report.md